# Notebook16 gated detached workflow
Definitions only. Default frontdoor Run All launches an immutable at-most12hour coordinator and deadline guard in tmux. Learned-token operator audits always produce an inspectable report; fresh confirmation requires an actual quality handoff, empirical gradient/compression gate and measured whole-JEPA speed gate. Inputs are frozen training checkpoints and audited memberships; official test stays closed until a source-bound seal. Outputs live in artifacts/notebook_16/<run_id> and checkpoints/<run_id>. Exact resume: fresh kernel, same RUN_ID, frontdoor16 Run All; inspect invalid/IN_FLIGHT journals, never reset a clock or silently restart.

In [ ]:
"""Executable, detached, gated hierarchy benchmark and fresh confirmation coordinator."""
from pathlib import Path
import copy,html,json,os,shlex,signal,subprocess,time,traceback
import nbformat
_SELF_PATH=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/16_hierarchical_confirmation/workflow.ipynb'))
_COMMON_PATH=_SELF_PATH.with_name('common.ipynb');_namespace={'__file__':str(_COMMON_PATH)}
for _cell in nbformat.read(_COMMON_PATH,4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_COMMON_PATH),'exec'),_namespace)
C=type('Common',(),_namespace)
DEFAULT_HIERARCHY={'leaf_size':32,'max_rank':16,'tolerance':1e-4,'eta':1.}
TERMINAL={'COMPLETE','NO_GO','UNKNOWN','FAILED','VALIDATION_COMPLETE'}
MAIN='notebooks/16_Hierarchical_Scaling_and_Confirmation.ipynb'

def artifact_path(root,run_id):
    if not isinstance(run_id,str)or not run_id or any(not(c.isalnum()or c in'-_')for c in run_id):raise ValueError('Safe stable RUN_ID required')
    return Path(root).resolve()/'artifacts/notebook_16'/run_id

def session_names(run_id):
    prefix='fmm16-'+run_id;return {'coordinator':prefix,'guard':prefix+'-guard','worker0':prefix+'-worker0','worker1':prefix+'-worker1'}
def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0

def terminate_session_tree(session):
    """Stop only processes descending from this exact project-owned tmux session."""
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)

def validate_manifest_rotation(runner,directory):
    for attempt in range(4):
        try:return runner.validate_training_manifest(directory)
        except (FileNotFoundError,ValueError):
            if attempt==3:raise
            time.sleep(.25)

def consumer_hash(root):return C.digest([''.join(c['source'])for c in json.loads((Path(root)/MAIN).read_text())['cells']if c['cell_type']=='code'])
def check_registration(root,artifact):
    req=C.read(Path(artifact)/'launch_registration.json')
    if req['sources']!=C.source_hashes(root)or req['consumer_hash']!=consumer_hash(root):raise ValueError('Frozen16 source/frontdoor changed; resume refused')
    clock=C.read(Path(artifact)/'budget_clock.json')
    if req['clock']!=clock:raise ValueError('Original12h clock changed; no reset')
    return req,clock

def status(artifact,state,phase,**details):
    value={'status':state,'phase':phase,'updated_unix':time.time(),**details};C.atomic_json(Path(artifact)/'study_status.json',value);print(state,phase,flush=True);return value

def write_operator_notebook(root,path,function,arguments):
    code="from pathlib import Path\nimport json,nbformat,types,os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nROOT=Path("+repr(str(Path(root).resolve()))+")\nP=ROOT/'experiments/16_hierarchical_confirmation/workflow.ipynb'\nW=types.ModuleType('fmm16_detached_workflow');W.__file__=str(P)\nfor cell in nbformat.read(P,4).cells:\n    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)\nRESULT=W."+function+"(**json.loads("+repr(json.dumps(arguments))+"))\nprint(json.dumps(RESULT,indent=2,default=str))\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached16 '+function+'\nGenerated operator entrypoint. Frozen parent source, original deadline and single-writer checkpoints remain authoritative. Resume through Notebook16 with the same RUN_ID.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR FMM-JEPA','language':'python'}})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=nbformat.read(path,4)
        if [c.source for c in old.cells if c.cell_type=='code']!=[c.source for c in nb.cells if c.cell_type=='code']:raise ValueError('Detached entrypoint code changed')
    else:nbformat.write(nb,path)
    return path

def spawn(root,path,session,log):
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    jupyter=Path(root).parent/'.venv/bin/jupyter';command=' '.join(shlex.quote(str(x))for x in[jupyter,'nbconvert','--to','notebook','--execute','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',path])+' >> '+shlex.quote(str(log))+' 2>&1'
    env='CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),env+command],check=True)
    return {'status':'STARTED','session':session,'operator_notebook':str(path),'log':str(log)}

def render_report(root,artifact,result,benchmark=None,cost=None):
    """Render measured evidence only; missing values stay undefined."""
    import base64
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True);figures=[]
    colors={'vit':'#315f8c','sdpa':'#315f8c','hierarchical':'#c27228','optional_geometry':'#c27228'}
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    fixture=bool(result.get('fixture_source')or(benchmark or {}).get('profile')=='validation')
    prefix='Software fixture — 'if fixture else''
    def save(fig,name):
        fig.savefig(artifact/name,dpi=150,bbox_inches='tight');plt.close(fig);figures.append(name)
    if benchmark and benchmark.get('rows'):
        rows=benchmark['rows'];counts=sorted({r['tokens']for r in rows});fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
        def aggregate(fn,complete=True):
            values=[]
            for n in counts:
                raw=[fn(r)for r in rows if r['tokens']==n]
                finite=[float(x)for x in raw if x is not None and np.isfinite(float(x))]
                values.append(max(finite)if finite and(not complete or len(finite)==len(raw))else np.nan)
            return values
        axes[0,0].plot(counts,aggregate(lambda r:r['measurements']['hierarchical']['forward_backward_seconds']/r['measurements']['sdpa']['forward_backward_seconds']),'o-',color=colors['hierarchical'])
        axes[0,0].axhline(1,color='#555555',linestyle='--');axes[0,0].set_ylabel('Hierarchy / SDPA time (ratio)');axes[0,0].set_ylim(bottom=0);axes[0,0].set_title('Worst measured operator forward + backward')
        for key,label,marker,color in [('relative_output_l2','Output','o',colors['sdpa']),('max_relative_gradient_l2','Worst gradient','s',colors['hierarchical'])]:
            axes[0,1].plot(counts,aggregate(lambda r:r['errors']['hierarchical'][key]),marker+'-',label=label,color=color)
        axes[0,1].set_yscale('symlog',linthresh=1e-7);axes[0,1].set_ylabel('Relative L2 error');axes[0,1].set_title('Empirical output and gradient error');axes[0,1].legend()
        axes[1,0].plot(counts,aggregate(lambda r:r['measurements']['hierarchical']['operator']['entry_work_over_dense']),'o-',color=colors['hierarchical'])
        axes[1,0].axhline(1,color='#555555',linestyle='--');axes[1,0].set_ylabel('Kernel-entry evaluations / N²');axes[1,0].set_ylim(bottom=0);axes[1,0].set_title('Sampled far entries and exact near entries')
        memory_measured=False
        for implementation,label,marker in [('sdpa','SDPA','o'),('hierarchical','Hierarchy','s')]:
            values=aggregate(lambda r:None if r['measurements'][implementation].get('peak_cuda_allocated_bytes')is None else r['measurements'][implementation]['peak_cuda_allocated_bytes']/1024**3)
            memory_measured=memory_measured or any(np.isfinite(values))
            axes[1,1].plot(counts,values,marker+'-',label=label,color=colors[implementation])
        if memory_measured:
            axes[1,1].set_ylim(bottom=0);axes[1,1].set_ylabel('Peak allocated CUDA memory (GiB)');axes[1,1].set_title('Worst measured CUDA peak; gaps are UNKNOWN');axes[1,1].legend()
        else:
            axes[1,1].clear();axes[1,1].text(.5,.5,'CUDA memory: NOT MEASURED\nCPU software validation',ha='center',va='center',transform=axes[1,1].transAxes);axes[1,1].set_axis_off()
        for ax in axes.flat:
            if ax.axison:ax.set_xlabel('Tokens N');ax.set_xscale('log',base=2);ax.set_xticks(counts,[str(n)for n in counts]);ax.grid(alpha=.2)
        fig.suptitle(prefix+'Frozen training-image kernel audit');save(fig,'operator_audit.png')
    paired=result.get('paired_results',[])
    if paired:
        fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
        for j,key,label in [(0,'vit_percent','ViT'),(1,'candidate_percent','Hierarchical candidate')]:
            values=[float(row[key])for row in paired]
            axes[0].scatter(np.full(len(values),j),values,facecolors='none',edgecolors=colors['vit'if j==0 else'optional_geometry'],s=48)
            axes[0].scatter([j],[np.mean(values)],marker='D',color=colors['vit'if j==0 else'optional_geometry'],s=85)
        axes[0].set_xticks([0,1],['ViT','Hierarchical candidate']);axes[0].set_ylim(0,max(1.,max(float(row[key])for row in paired for key in('vit_percent','candidate_percent')))*1.08);axes[0].set_ylabel('Official-subset top-1 accuracy (%)');axes[0].set_title(f'{len(paired)} fresh paired training seeds')
        values=[float(row['delta_pp'])for row in paired];indices=np.arange(1,len(values)+1)
        axes[1].scatter(indices,values,color=colors['optional_geometry']);axes[1].axhline(0,color='#444444',linewidth=1);axes[1].axhline(result.get('minimum_effect_pp',.5),color='#8a7640',linestyle='--',label='Registered minimum mean effect')
        axes[1].axhline(result['mean_delta_pp'],color=colors['optional_geometry'],linestyle=':',label=f"Mean {result['mean_delta_pp']:+.3f} pp")
        ci=result.get('raw_ci95_pp')
        if ci is not None:axes[1].axhspan(ci[0],ci[1],color=colors['optional_geometry'],alpha=.14,label='95% paired-t CI for the mean')
        else:axes[1].text(.02,.98,'Paired-t CI undefined',transform=axes[1].transAxes,va='top')
        axes[1].set_xlabel('Independent paired seed index');axes[1].set_ylabel('Candidate − ViT (percentage points)');axes[1].set_title('All paired effects and mean uncertainty');axes[1].legend(fontsize=9)
        fig.suptitle(prefix+'Frozen official-subset confirmation — fixed converged readout');save(fig,'confirmation_quality.png')
    training_path=artifact/'training_result.json'
    if training_path.exists():
        values=list(C.read(training_path).get('arms',{}).values())
        values=[v for v in values if isinstance(v.get('active_wall_seconds'),(int,float))and isinstance(v.get('actual_updates'),(int,float))]
        if values:
            fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
            for j,variant in enumerate(('vit','optional_geometry')):
                selected=[v for v in values if v['variant']==variant]
                for ax,key in zip(axes,['active_wall_seconds','actual_updates']):
                    if selected:ax.scatter(np.full(len(selected),j),[v[key]for v in selected],facecolors='none',edgecolors=colors[variant],s=45)
            for ax in axes:ax.set_xticks([0,1],['ViT','Hierarchical candidate']);ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.2)
            axes[0].set_ylabel('Recorded active training wall time (seconds)');axes[0].set_title('Every completed timed arm')
            axes[1].set_ylabel('Successful JEPA optimizer updates');axes[1].set_title('Updates achieved within the recorded budget')
            plan_path=artifact/'confirmation_plan.json'
            if plan_path.exists():axes[0].axhline(C.read(plan_path)['active_wall_seconds_per_arm'],color='#777777',linestyle='--',label='Registered time budget');axes[0].legend()
            target=float(C.read(plan_path)['active_wall_seconds_per_arm'])if plan_path.exists()else 0.
            axes[0].set_ylim(0,max(1.,target,max(float(v['active_wall_seconds'])for v in values))*1.08)
            axes[1].set_ylim(0,max(1.,max(float(v['actual_updates'])for v in values))*1.08)
            fig.suptitle(prefix+'Recorded training budgets and update counts');save(fig,'training_budget.png')
    reasons=result.get('reason',result.get('decision','Inspect source-bound gate receipts'))
    body=['<!doctype html><html><head><meta charset="utf-8"><title>FMM JEPA Notebook16</title><style>body{font:16px system-ui;max-width:1100px;margin:40px auto;padding:0 20px;color:#17232d}pre{overflow:auto;background:#edf2f5;padding:18px}img{width:100%;height:auto}table{border-collapse:collapse;width:100%;font-size:14px}th,td{padding:8px;text-align:left;border-bottom:1px solid #dbe2e8}details{margin:20px 0}h2{margin-top:28px}</style></head><body>',
      '<h1>Hierarchical scaling and confirmation</h1><p><strong>'+html.escape(result['status'])+'</strong> — '+html.escape(str(reasons))+'</p>',
      '<p>'+('SOFTWARE VALIDATION ONLY. These fixtures do not establish a scientific gain.'if fixture else'Measured evidence from the registered run; completed cases and every paired seed are retained.')+'</p>',
      '<p>This is hierarchical kernel summation using adaptive cross approximation, inspired by FMM. It does not implement classical multipole expansion and translation operators. Sampled residual and negative-entry checks do not prove uniform error or global positivity. Whole-output and parameter-gradient comparisons are empirical; pivots and topology are piecewise controls.</p>',
      '<p>Official test: '+html.escape(str(result.get('official_test','CLOSED')))+'. Missing quality evidence or failed cost/error gates keep confirmation closed. Incomplete endpoints remain UNKNOWN.</p>']
    if paired:
        ci=result.get('raw_ci95_pp');ci_text='undefined'if ci is None else f'[{ci[0]:+.3f}, {ci[1]:+.3f}] pp'
        body.append('<h2>Registered accuracy comparison</h2><p>Mean candidate − ViT: <strong>'+format(result['mean_delta_pp'],'+.3f')+' pp</strong>; paired 95% t interval '+ci_text+'. Training-seed pairs are the inferential units; image count is not replicate count.</p>')
    for name in figures:
        data=base64.b64encode((artifact/name).read_bytes()).decode();body.append('<img alt="'+html.escape(name.replace('_',' ').replace('.png',''))+'" src="data:image/png;base64,'+data+'">')
    if benchmark:
        body.append('<h2>Every audited operator case</h2><table><tr><th>Case</th><th>N</th><th>Hierarchy / SDPA time</th><th>Output error</th><th>Worst gradient error</th><th>Maximum far rank</th><th>Entry work / N²</th></tr>')
        for row in benchmark.get('rows',[]):
            op=row['measurements']['hierarchical']['operator'];body.append('<tr><td>'+html.escape(row['key'])+'</td><td>'+str(row['tokens'])+'</td><td>'+format(1/row['hierarchical_speed_over_sdpa'],'.3g')+'</td><td>'+format(row['errors']['hierarchical']['relative_output_l2'],'.3g')+'</td><td>'+format(row['errors']['hierarchical']['max_relative_gradient_l2'],'.3g')+'</td><td>'+str(op.get('accepted_far_rank_max'))+'</td><td>'+format(op['entry_work_over_dense'],'.3g')+'</td></tr>')
        body.append('</table>')
    body.append('<h2>Gate decisions</h2><table><tr><th>Gate</th><th>Status</th><th>Scope</th></tr>')
    for gate in result.get('gate_receipts',[]):body.append('<tr><td>'+html.escape(gate.get('role',''))+'</td><td>'+html.escape(gate.get('status','UNKNOWN'))+'</td><td>'+html.escape(gate.get('scope','See saved evidence receipt'))+'</td></tr>')
    body.append('</table>')
    if cost:
        body.append('<h2>Measured complete JEPA step</h2><p>Image size '+str(cost.get('image_size'))+' px; effective batch '+str(cost.get('effective_batch'))+'. End-to-end speed relative to optimized ViT: '+format(cost['measured_end_to_end_speed_over_optimized_vit'],'.3f')+'×. Real decoding, augmentation, masks, training and EMA are included.</p>')
        body.append('<details><summary>Whole JEPA timing receipt</summary><pre>'+html.escape(json.dumps(cost,indent=2))+'</pre></details>')
    body.append('<details><summary>Complete result and audit receipts</summary><pre>'+html.escape(json.dumps(result,indent=2,default=str))+'</pre></details>')
    body.append('<p>Source evidence: <a href="result.json">result.json</a>; operator and confirmation receipts are saved beside this report. Images are embedded for portable review.</p></body></html>')
    (artifact/'report.html').write_text('\n'.join(body));C.atomic_json(artifact/'result.json',result)
    return {'html':str(artifact/'report.html'),'figures':figures,'result':str(artifact/'result.json')}

def run_validation(root,run_id='hierarchical-validation'):
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    B=C.load(root,'experiments/16_hierarchical_confirmation/benchmark.ipynb','fmm16_benchmark')
    bench=B.compression_benchmark(root,artifact,{'status':'NOT_AVAILABLE','fixture_source':'SOFTWARE_ONLY'},DEFAULT_HIERARCHY,time.time()+180,profile='validation')
    if not bench['error_gate']:raise AssertionError('Software fixture operator/gradient audit failed')
    result={'status':'VALIDATION_COMPLETE','fixture_source':'SYNTHETIC_SOFTWARE_CHECK; NOT SCIENTIFIC RESULTS','operator_error_pass':bench['error_gate'],
        'confirmation_launched':False,'reason':'RunAll validation exercises operator measurement, empirical errors, checkpoint journals separately, source binding and visual report. Official test and long runs remain closed.','official_test':'CLOSED','sources':C.source_hashes(root)}
    render_report(root,artifact,result,bench);status(artifact,'VALIDATION_COMPLETE','software_validation',report=str(artifact/'report.html'));return result

def launch_study(root,run_id='hierarchical-confirmation-001',profile=None,source_run=None,hierarchy=None,max_seconds=43200):
    root=Path(root).resolve();C.bootstrap(root);profile=profile or os.environ.get('FMM16_PROFILE','full')
    if profile=='validation':return run_validation(root,run_id)
    if profile!='full':raise ValueError('Registered full or validation profile required')
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True);hierarchy=dict(hierarchy or DEFAULT_HIERARCHY)
    from fmm_jepa.food101_training import run_writer
    with run_writer(artifact/'launch_writer'):
        regpath=artifact/'launch_registration.json';clockpath=artifact/'budget_clock.json'
        if regpath.exists():
            req,clock=check_registration(root,artifact)
            if req['hierarchy']!=hierarchy or req['source_run']!=source_run:raise ValueError('Launch request changed for existing RUN_ID')
        else:
            if clockpath.exists():raise ValueError('Orphan budget clock requires inspection; no restart')
            seconds=min(43200,float(max_seconds))
            if seconds<=0:raise ValueError('Positive bounded12h clock required')
            start=time.time();clock={'started_unix':start,'deadline_unix':start+seconds,'maximum_seconds':seconds,'scope':'Operator extraction/benchmark, software exact-resume preflights, quality gates, fresh equal-active-wall training and final sealed evaluation; immutable across restart'}
            C.atomic_json(clockpath,clock)
            req={'run_id':run_id,'profile':'full','clock':clock,'sources':C.source_hashes(root),'consumer_hash':consumer_hash(root),'hierarchy':hierarchy,'source_run':source_run,'handoff_snapshot':C.handoff(root,source_run),'official_test':'CLOSED'}
            C.immutable(regpath,req)
        existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if existing.get('status')in TERMINAL:return {**existing,'report':str(artifact/'report.html'),'resume':'Terminal protocol receipt retained; new scientific design uses a new RUN_ID'}
        C.check_deadline(clock['deadline_unix'])
        sessions=session_names(run_id);guard=write_operator_notebook(root,artifact/'operators/guard.ipynb','run_guard',{'root':str(root),'run_id':run_id})
        coordinator=write_operator_notebook(root,artifact/'operators/coordinator.ipynb','run_coordinator',{'root':str(root),'run_id':run_id})
        guards=spawn(root,guard,sessions['guard'],artifact/'guard.log');job=spawn(root,coordinator,sessions['coordinator'],artifact/'coordinator.log')
        value={'status':'DETACHED_PIPELINE_STARTED','run_id':run_id,'sessions':sessions,'clock':clock,'artifact':str(artifact),
            'attach':'tmux attach -t '+sessions['coordinator'],'detach':'Ctrl-b, then d','resume':'Fresh project kernel, same RUN_ID, Notebook16 Run All. Original clock/source bindings and full-state checkpoints are rechecked.','report':str(artifact/'report.html'),'guard':guards,'coordinator':job}
        C.atomic_json(artifact/'launch_receipt.json',value);return value

def run_worker(root,run_id,worker):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);folder=artifact/'workers'/worker
    T=C.load(root,'experiments/16_hierarchical_confirmation/runner.ipynb','fmm16_timed_runner');plan=C.read(folder/'plan.json')
    try:
        status(folder,'RUNNING','data_verification');C.environment(plan['arms'][0]['config']);bundle=C.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=False)
        reserve=float(C.read(artifact/'confirmation_proposal.json')['evaluation_reserve_seconds']);training_deadline=clock['deadline_unix']-reserve
        status(folder,'RUNNING','exact_resume_and_timed_training');training=T.train_worker(root,folder,plan,bundle,training_deadline)
        if training['status']!='COMPLETE':status(folder,'UNKNOWN','training_fairness');return training
        status(folder,'RUNNING','converged_development_readout');evaluation=T.evaluate_worker(root,folder,plan,bundle,training,clock['deadline_unix'])
        state='COMPLETE'if evaluation['status']=='COMPLETE'else'UNKNOWN';status(folder,state,'worker_finished');return {'status':state,'training':str(folder/'training_result.json'),'evaluation':str(folder/'evaluation_result.json')}
    except BaseException as exc:
        state='UNKNOWN'if isinstance(exc,TimeoutError)else'FAILED';status(folder,state,'worker_exception',reason=repr(exc),traceback=traceback.format_exc());raise

def run_coordinator(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);deadline=clock['deadline_unix']
    from fmm_jepa.food101_training import run_writer
    bench=None;cost=None
    try:
        with run_writer(artifact/'coordinator_writer'):
            C.check_deadline(deadline,60);status(artifact,'RUNNING','learned_token_compression')
            B=C.load(root,'experiments/16_hierarchical_confirmation/benchmark.ipynb','fmm16_benchmark')
            bench=B.compression_benchmark(root,artifact,req['handoff_snapshot'],req['hierarchy'],deadline,profile='full')
            operator={'role':'operator','status':'PASS'if bench['confirmation_eligible']else'FAIL','passed':bench['confirmation_eligible'],'evidence':C.file_record(artifact/'compression_benchmark.json'),
                'scope':'Actual learned-token empirical output/all-parameter-gradient errors, sampled work and measured forward/backward vs observed TorchSDPA'}
            handoff=req['handoff_snapshot'];quality={'role':'quality','status':'PASS'if handoff.get('status')=='COMPLETE'and handoff.get('quality_pass')and not handoff.get('fixture_source')else'FAIL','passed':bool(handoff.get('quality_pass')),
                'evidence':handoff.get('handoff_record'),'scope':'Notebook15 predeclared four paired seeds quality handoff'}
            receipts=[quality,operator]
            if operator['status']!='PASS' or quality['status']!='PASS':
                result={'status':'NO_GO','reason':'Confirmation remains closed: '+('learned-token compression/error/cost gate failed'if operator['status']!='PASS'else'Notebook15 scientific quality handoff unavailable or did not pass'),
                    'gate_receipts':receipts,'full_jepa_gate':'NOT_RUN_AFTER_PRECEDING_GATE_FAILURE','official_test':'CLOSED','source_hashes':req['sources']}
                render_report(root,artifact,result,bench);status(artifact,'NO_GO','gates_closed',report=str(artifact/'report.html'));return result
            C.check_deadline(deadline,3600);status(artifact,'RUNNING','actual_full_jepa_cost')
            cost=B.full_jepa_speed_gate(root,artifact,handoff,req['hierarchy'],deadline,image_size=256,microbatch=32)
            receipts.append({'role':'full_jepa_cost','status':cost['status'],'passed':cost['status']=='PASS','evidence':C.file_record(artifact/'full_jepa_speed_gate.json')})
            if cost['status']!='PASS':
                result={'status':'NO_GO','reason':'Measured full-JEPA end-to-end wall speed did not exceed optimized ViT by registered10%','gate_receipts':receipts,'official_test':'CLOSED'}
                render_report(root,artifact,result,bench,cost);status(artifact,'NO_GO','full_jepa_cost_gate');return result
            F=C.load(root,'experiments/16_hierarchical_confirmation/confirmation.ipynb','fmm16_confirmation')
            proposal=F.prepare_registration(root,artifact,handoff,bench,deadline-time.time())
            if proposal['status']!='READY_FOR_FROZEN_CONFIRMATION':
                result={**proposal,'status':'UNKNOWN','reason':'Fresh paired power/budget requirement not feasible within remaining immutable12h','gate_receipts':receipts,'official_test':'CLOSED'}
                render_report(root,artifact,result,bench,cost);status(artifact,'UNKNOWN','power_budget_gate');return result
            T=C.load(root,'experiments/16_hierarchical_confirmation/runner.ipynb','fmm16_timed_runner');T.bind(root,req['hierarchy'])
            planpath=artifact/'confirmation_plan.json'
            plan=C.read(planpath)if planpath.exists()else T.make_plan(root,run_id,proposal,handoff,receipts,req['hierarchy'])
            if plan['source_hashes']!=req['sources']:raise ValueError('Confirmation sources changed since pipeline registration')
            C.immutable(planpath,plan);C.environment(plan['arms'][0]['config']);bundle=C.data_bundle(root,artifact/'data',plan['arms'][0]['config'],verify_images=True)
            F.register_confirmation(root,artifact,plan,handoff['screening']['paired_seed_deltas_pp'],bundle)
            sessions=session_names(run_id);status(artifact,'RUNNING','registered_paired_equal_wall_training',fresh_pairs=len(plan['seeds']))
            for worker in('worker0','worker1'):
                folder=artifact/'workers'/worker;folder.mkdir(parents=True,exist_ok=True);child=T.worker_plan(plan,worker);C.immutable(folder/'plan.json',child)
                C.immutable(folder/'launch_time.json',{'started_unix':C.read(folder/'launch_time.json')['started_unix']if(folder/'launch_time.json').exists()else time.time()})
                notebook=write_operator_notebook(root,folder/'worker.ipynb','run_worker',{'root':str(root),'run_id':run_id,'worker':worker});spawn(root,notebook,sessions[worker],folder/'worker.log')
            while True:
                C.check_deadline(deadline);states={w:C.read(artifact/'workers'/w/'study_status.json')if(artifact/'workers'/w/'study_status.json').exists()else{'status':'STARTING'}for w in('worker0','worker1')}
                if any(v['status']in('FAILED','UNKNOWN')for v in states.values()):raise RuntimeError('Worker failed/cutoff; scientific confirmation UNKNOWN:'+str({k:v['status']for k,v in states.items()}))
                if all(v['status']=='COMPLETE'for v in states.values()):break
                for worker,state in states.items():
                    if state['status']in('RUNNING','STARTING')and time.time()-C.read(artifact/'workers'/worker/'launch_time.json')['started_unix']>90 and not session_exists(sessions[worker]):raise RuntimeError('Worker session disappeared; checkpoints retained, UNKNOWN fairness pending inspection')
                C.atomic_json(artifact/'coordinator_heartbeat.json',{'updated_unix':time.time(),'worker_states':states,'deadline_unix':deadline});time.sleep(10)
            training={'status':'COMPLETE','fair_wall_budget_verified':True,'arms':{},'gate_receipts':receipts}
            devrows=[]
            for worker in('worker0','worker1'):
                child=C.read(artifact/'workers'/worker/'training_result.json');training['arms'].update(child['arms']);devrows.extend(C.read(artifact/'workers'/worker/'evaluation_result.json')['rows'])
            values=list(training['arms'].values());tolerance=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
            if len(values)!=len(plan['arms'])or max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)>tolerance:raise RuntimeError('Registered global equal-wall tolerance failed; final test remains closed')
            C.atomic_json(artifact/'training_result.json',training);C.atomic_json(artifact/'evaluation_result.json',{'status':'COMPLETE','rows':devrows,'official_test':'CLOSED'})
            status(artifact,'RUNNING','sealed_official_confirmation')
            result=F.seal_and_score(root,artifact,plan,bundle,training,{'model_builder':C.builder(root,req['hierarchy']),'rows':devrows},deadline)
            result.update(gate_receipts=receipts,source_hashes=req['sources']);render_report(root,artifact,result,bench,cost);status(artifact,result['status'],'finished',report=str(artifact/'report.html'));return result
    except BaseException as exc:
        for worker in('worker0','worker1'):
            name=session_names(run_id)[worker]
            if session_exists(name):terminate_session_tree(name)
        result={'status':'UNKNOWN','reason':repr(exc),'traceback':traceback.format_exc(),'official_test':'CLOSED_UNLESS_EXISTING_SEAL_AUTHORIZED','source_hashes':req['sources']}
        render_report(root,artifact,result,bench,cost);status(artifact,'UNKNOWN','cutoff_or_failure',reason=repr(exc));return result

def run_guard(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);sessions=session_names(run_id);last_review=0.
    coordinator_missing_since=None
    try:
        req=C.read(artifact/'launch_registration.json');clock=C.read(artifact/'budget_clock.json')
        while True:
            check_registration(root,artifact)
            existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if existing.get('status')in TERMINAL:return {'status':'GUARD_COMPLETE','study_status':existing['status']}
            now=time.time()
            if not session_exists(sessions['coordinator']):
                if coordinator_missing_since is None:coordinator_missing_since=now
                elif now-coordinator_missing_since>90:raise RuntimeError('Coordinator disappeared beyond startup grace; stop workers and preserve UNKNOWN')
            else:coordinator_missing_since=None
            C.atomic_json(artifact/'guard_heartbeat.json',{'status':'HEALTHY','updated_unix':now,'deadline_unix':clock['deadline_unix'],'sessions':{k:session_exists(v)for k,v in sessions.items()}})
            if now>=clock['deadline_unix']:raise TimeoutError('Original immutable12h deadline reached')
            if now-last_review>=3600:
                audits=[];R,_,_=C.modules(root)
                for manifest in (root/'checkpoints'/run_id).glob('stages/worker*/arms/*/manifest.json'):
                    m=validate_manifest_rotation(R,manifest.parent);audits.append({'directory':str(manifest.parent),'successful_updates':m['successful_updates'],'attempted_updates':m['attempted_updates'],'latest_valid':True})
                C.atomic_json(artifact/'hourly_review.json',{'status':'PASS','updated_unix':now,'audits':audits,'source_bindings_valid':True,'official_test':'No predictive reader in guard; seal governs opening'});last_review=now
            time.sleep(10)
    except BaseException as exc:
        # Stop coordinator before publishing the sole terminal report, then all workers and kernel descendants.
        stopped={}
        for role in('coordinator','worker0','worker1'):stopped[role]=terminate_session_tree(sessions[role])
        result={'status':'UNKNOWN','reason':repr(exc),'stopped_project_processes':stopped,'official_test':'CLOSED_UNLESS_EXISTING_SEAL_AUTHORIZED','clock':locals().get('clock',{'status':'UNREADABLE_ORIGINAL_CLOCK'})}
        bench=C.read(artifact/'compression_benchmark.json')if(artifact/'compression_benchmark.json').exists()else None
        render_report(root,artifact,result,bench);status(artifact,'UNKNOWN','deadline_or_guard_integrity_failure',reason=repr(exc));return result
